In [ ]:
# ============================================================
# NOTEBOOK 08A-BIS — DIAGNOSTIC ANALYSIS
# NO MODEL RETRAINING
#
# Reads saved outputs from 08A and produces:
# - Train / Validation / Test generalization comparison
# - Per-device TEST diagnostics
# - Error quantiles and large-error rates
# - Performance by magnitude of actual temperature change
# - Temporal error summaries
# - Feature coefficient / importance summaries
# - Paired TEST loss differences vs Persistence
#
# This is diagnostic/descriptive analysis only.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
IN = ROOT / "outputs" / "08A_tabular_models"
OUT = ROOT / "outputs" / "08A_bis_diagnostics"
OUT.mkdir(parents=True, exist_ok=True)

METRICS = IN / "04_metrics_all_splits.csv"
PRED = IN / "05_predictions_all_models.csv"
PER_DEVICE = IN / "07_TEST_metrics_per_device.csv"
RIDGE_COEF = IN / "09_ridge_coefficients.csv"
RF_IMP = IN / "10_rf_feature_importance.csv"
XGB_IMP = IN / "11_xgb_feature_importance.csv"

for p in [METRICS, PRED, PER_DEVICE, RIDGE_COEF, RF_IMP, XGB_IMP]:
    assert p.exists(), f"Missing: {p}"

metrics = pd.read_csv(METRICS)
pred = pd.read_csv(PRED)
pred["timestamp"] = pd.to_datetime(pred["timestamp"], utc=True)
pred["target_timestamp"] = pd.to_datetime(pred["target_timestamp"], utc=True)

models = ["Persistence", "Ridge", "RandomForest", "XGBoost"]
splits = ["train", "validation", "test"]

# ------------------------------------------------------------
# 1. TRAIN -> VALIDATION -> TEST GENERALIZATION
# ------------------------------------------------------------
g = metrics[metrics["model"].isin(models)].copy()
g["RMSE_gap_vs_train_pct"] = np.nan
g["MAE_gap_vs_train_pct"] = np.nan

for model in models:
    tr = g[(g.model == model) & (g.split == "train")].iloc[0]
    mask = g.model == model
    g.loc[mask, "RMSE_gap_vs_train_pct"] = (
        100 * (g.loc[mask, "RMSE"] - tr["RMSE"]) / tr["RMSE"]
    )
    g.loc[mask, "MAE_gap_vs_train_pct"] = (
        100 * (g.loc[mask, "MAE"] - tr["MAE"]) / tr["MAE"]
    )

g.to_csv(OUT / "01_generalization_train_validation_test.csv", index=False)

# ------------------------------------------------------------
# 2. TEST ERROR DISTRIBUTIONS
# ------------------------------------------------------------
test = pred[pred["split"] == "test"].copy()
test["abs_error"] = test["error"].abs()
test["sq_error"] = test["error"] ** 2

dist_rows = []
for model, d in test.groupby("model"):
    ae = d["abs_error"].to_numpy()
    dist_rows.append({
        "model": model,
        "n": len(d),
        "median_abs_error": np.median(ae),
        "p75_abs_error": np.quantile(ae, .75),
        "p90_abs_error": np.quantile(ae, .90),
        "p95_abs_error": np.quantile(ae, .95),
        "p99_abs_error": np.quantile(ae, .99),
        "max_abs_error": np.max(ae),
        "pct_abs_error_gt_0_1C": 100*np.mean(ae > .1),
        "pct_abs_error_gt_0_2C": 100*np.mean(ae > .2),
        "pct_abs_error_gt_0_5C": 100*np.mean(ae > .5),
    })
dist = pd.DataFrame(dist_rows).sort_values("median_abs_error")
dist.to_csv(OUT / "02_TEST_error_distribution.csv", index=False)

# ------------------------------------------------------------
# 3. ACTUAL TEMPERATURE CHANGE AND DIFFICULTY STRATA
# ------------------------------------------------------------
# Persistence prediction equals T(t), so:
# actual_delta = y_true - T(t) = y_true - persistence_prediction.
pers = test[test["model"] == "Persistence"][
    ["uuid", "timestamp", "y_true", "y_pred"]
].rename(columns={"y_pred": "temperature_t"})

pers["actual_delta_T_15min"] = pers["y_true"] - pers["temperature_t"]
pers["abs_delta_T_15min"] = pers["actual_delta_T_15min"].abs()

def change_bin(x):
    if x < 0.02: return "<0.02°C"
    if x < 0.05: return "0.02–0.05°C"
    if x < 0.10: return "0.05–0.10°C"
    if x < 0.20: return "0.10–0.20°C"
    return ">=0.20°C"

pers["change_bin"] = pers["abs_delta_T_15min"].map(change_bin)
pers.to_csv(OUT / "03_TEST_actual_temperature_change.csv", index=False)

joined = test.merge(
    pers[["uuid","timestamp","abs_delta_T_15min","change_bin"]],
    on=["uuid","timestamp"], how="left", validate="many_to_one"
)

difficulty = []
order = ["<0.02°C","0.02–0.05°C","0.05–0.10°C","0.10–0.20°C",">=0.20°C"]
for (model, b), d in joined.groupby(["model","change_bin"]):
    difficulty.append({
        "model": model,
        "change_bin": b,
        "n": len(d),
        "MAE": d["abs_error"].mean(),
        "RMSE": np.sqrt(d["sq_error"].mean()),
        "Bias": d["error"].mean(),
    })
difficulty = pd.DataFrame(difficulty)
difficulty["change_bin"] = pd.Categorical(
    difficulty["change_bin"], categories=order, ordered=True
)
difficulty = difficulty.sort_values(["change_bin","model"])
difficulty.to_csv(OUT / "04_TEST_metrics_by_temperature_change.csv", index=False)

change_summary = pd.DataFrame({
    "statistic": ["mean_abs_change","median_abs_change","p90_abs_change",
                  "p95_abs_change","p99_abs_change","max_abs_change"],
    "value_C": [
        pers["abs_delta_T_15min"].mean(),
        pers["abs_delta_T_15min"].median(),
        pers["abs_delta_T_15min"].quantile(.90),
        pers["abs_delta_T_15min"].quantile(.95),
        pers["abs_delta_T_15min"].quantile(.99),
        pers["abs_delta_T_15min"].max(),
    ]
})
change_summary.to_csv(OUT / "05_TEST_temperature_change_summary.csv", index=False)

# ------------------------------------------------------------
# 4. PER-DEVICE TEST PERFORMANCE + RANK
# ------------------------------------------------------------
pdv = pd.read_csv(PER_DEVICE)
pdv["rank_RMSE_within_uuid"] = (
    pdv.groupby("uuid")["RMSE"].rank(method="min")
)
pdv.sort_values(["uuid","RMSE"]).to_csv(
    OUT / "06_TEST_per_device_ranked.csv", index=False
)

winner = (
    pdv.sort_values(["uuid","RMSE"])
       .groupby("uuid", as_index=False)
       .first()[["uuid","model","RMSE","MAE","R2","Bias"]]
)
winner.to_csv(OUT / "07_TEST_best_model_per_device.csv", index=False)

# ------------------------------------------------------------
# 5. TEMPORAL ERROR — DAILY SUMMARY ON TEST
# ------------------------------------------------------------
test["date"] = test["timestamp"].dt.date.astype(str)
daily = (
    test.groupby(["model","date"])
    .agg(
        n=("error","size"),
        MAE=("abs_error","mean"),
        RMSE=("sq_error", lambda x: float(np.sqrt(x.mean()))),
        Bias=("error","mean")
    )
    .reset_index()
)
daily.to_csv(OUT / "08_TEST_daily_error.csv", index=False)

# ------------------------------------------------------------
# 6. FEATURE COEFFICIENTS / IMPORTANCES
# ------------------------------------------------------------
ridge = pd.read_csv(RIDGE_COEF)
ridge["abs_ridge_coefficient"] = ridge["ridge_coefficient"].abs()
ridge.sort_values("abs_ridge_coefficient", ascending=False).to_csv(
    OUT / "09_ridge_coefficients_ranked.csv", index=False
)

rf = pd.read_csv(RF_IMP).sort_values("rf_importance", ascending=False)
rf.to_csv(OUT / "10_rf_importance_ranked.csv", index=False)

xgb = pd.read_csv(XGB_IMP).sort_values("xgb_importance", ascending=False)
xgb.to_csv(OUT / "11_xgb_importance_ranked.csv", index=False)

# ------------------------------------------------------------
# 7. PAIRED TEST LOSS DIFFERENCES VS PERSISTENCE
# ------------------------------------------------------------
wide = test.pivot(
    index=["uuid","timestamp"], columns="model", values="abs_error"
)
assert wide.notna().all().all()

paired = []
p_loss = wide["Persistence"]
for model in ["Ridge","RandomForest","XGBoost"]:
    diff = wide[model] - p_loss
    paired.append({
        "model_vs_persistence": model,
        "n": len(diff),
        "mean_abs_error_difference_C": diff.mean(),
        "median_abs_error_difference_C": diff.median(),
        "pct_samples_model_better_than_persistence": 100*np.mean(diff < 0),
        "pct_samples_equal": 100*np.mean(np.isclose(diff, 0, atol=1e-12)),
        "pct_samples_model_worse_than_persistence": 100*np.mean(diff > 0),
    })
paired = pd.DataFrame(paired)
paired.to_csv(OUT / "12_TEST_paired_absolute_error_vs_persistence.csv", index=False)

# ------------------------------------------------------------
# 8. COMPACT CONSOLE REPORT
# ------------------------------------------------------------
print("="*78)
print("08A-BIS — DIAGNOSTIC ANALYSIS (NO RETRAINING)")
print("="*78)

print("\nTRAIN / VALIDATION / TEST")
print(
    g[["model","split","MAE","RMSE","R2","Bias","RMSE_gap_vs_train_pct"]]
    .sort_values(["model","split"])
    .to_string(index=False)
)

print("\nTEST ERROR DISTRIBUTION")
print(dist.to_string(index=False))

print("\nACTUAL |ΔT| OVER 15 MIN — TEST")
print(change_summary.to_string(index=False))

print("\nBEST MODEL PER UUID — TEST")
print(winner.to_string(index=False))

print("\nPAIRED ABSOLUTE ERROR VS PERSISTENCE — TEST")
print(paired.to_string(index=False))

print("\nTOP 5 RIDGE |COEFFICIENTS|")
print(ridge.sort_values("abs_ridge_coefficient", ascending=False).head(5).to_string(index=False))

print("\nTOP 5 RF IMPORTANCES")
print(rf.head(5).to_string(index=False))

print("\nTOP 5 XGB IMPORTANCES")
print(xgb.head(5).to_string(index=False))

print("\nVALIDATION")
print("✓ No model retraining")
print("✓ Uses saved 08A predictions only")
print("✓ TEST errors remain paired by UUID and timestamp")
print("✓ Persistence used to recover actual 15-min temperature change")
print("✓ Per-device and temporal diagnostics saved")
print("✓ Feature diagnostics saved")
print("✓ Ready for interpretation before 08B")
